# Cross-Template Key-Value Field Extraction — Kaggle Pilot

**IE 643 Course Project** · Pipeline sanity (Track A) and headline experiment (Track B)

| Track | Dataset | Protocol | Goal |
|---|---|---|---|
| **A** | FUNSD | Standard split | F1 ≈ 0.88 — validates the training/eval loop |
| **B** | VRDU Registration Forms | MTL vs UTL, matched size | Measure the generalization gap |

> **Runtime**: Kaggle GPU (P100 16 GB or 2×T4).  
> **Quota**: 30 hrs/week GPU.  
> Track A ≈ 1–2 hrs (3 seeds × 20 epochs).  Track B ≈ 3–5 hrs (3 folds × 2 regimes × 3 seeds × 20 epochs).

---

## 0 · Environment Setup

In [ ]:
# ── Clone the project repo ──────────────────────────────────────────────
# Replace with your actual repo URL (HTTPS or SSH)
REPO_URL = "https://github.com/<your-org>/KeyValue-Extraction.git"  # TODO: update
REPO_DIR = "/kaggle/working/KeyValue-Extraction"

import os, subprocess, sys

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"{REPO_DIR} already exists, skipping clone")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────
# Kaggle pre-installs torch, transformers, etc. — pip will skip them if
# the version constraint is already satisfied.
!pip install -q -r requirements.txt 2>&1 | tail -5

In [ ]:
# ── Verify GPU and key imports ──────────────────────────────────────────
import torch
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: no GPU — training will be very slow")

import transformers, datasets, seqeval
print(f"transformers {transformers.__version__}")
print(f"datasets {datasets.__version__}")

# Add src/ to the Python path
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print(f"\nsys.path includes: {os.path.join(REPO_DIR, 'src')}")

## 1 · Download Datasets

In [ ]:
# ── FUNSD ───────────────────────────────────────────────────────────────
# The raw FUNSD release (annotations + images) from the official source.
# We parse the raw release because the HF mirror drops `linking` edges.

FUNSD_DIR = os.path.join(REPO_DIR, "data", "funsd", "dataset")

if not os.path.isdir(FUNSD_DIR):
    print("Downloading FUNSD...")
    !mkdir -p data/funsd
    !wget -q https://guillaumejaume.github.io/FUNSD/dataset.zip -O data/funsd/dataset.zip
    !cd data/funsd && unzip -q dataset.zip && rm dataset.zip
    print("FUNSD extracted.")
else:
    print(f"FUNSD already at {FUNSD_DIR}")

# Verify expected paths exist
for split in ("training_data", "testing_data"):
    for sub in ("annotations", "images"):
        p = os.path.join(FUNSD_DIR, split, sub)
        n = len(os.listdir(p)) if os.path.isdir(p) else 0
        print(f"  {split}/{sub}: {n} files")

In [ ]:
# ── VRDU ────────────────────────────────────────────────────────────────
# Clone the VRDU benchmark repo (Registration Forms + Ad-Buy Forms).

VRDU_DIR = os.path.join(REPO_DIR, "data", "vrdu")

if not os.path.isdir(VRDU_DIR):
    print("Cloning VRDU...")
    !git clone --depth 1 https://github.com/google-research-datasets/vrdu.git data/vrdu
    print("VRDU cloned.")
else:
    print(f"VRDU already at {VRDU_DIR}")

# Verify expected paths
for name in ("registration-form", "ad-buy-form"):
    base = os.path.join(VRDU_DIR, name)
    data_gz = os.path.join(base, "main", "dataset.jsonl.gz")
    meta = os.path.join(base, "main", "meta.json")
    splits = os.path.join(base, "few_shot-splits")
    n_splits = len(os.listdir(splits)) if os.path.isdir(splits) else 0
    print(f"  {name}: data={'OK' if os.path.isfile(data_gz) else 'MISSING'}  "
          f"meta={'OK' if os.path.isfile(meta) else 'MISSING'}  "
          f"splits={n_splits} files")

---
## 2 · Track A — Pipeline Sanity: LiLT on FUNSD

**Purpose**: Prove the training/eval loop is correct.  Target F1 ≈ 0.88.

> ⚠️ **This is NOT evidence of generalization.**  
> FUNSD's train/test share 16% template overlap (Laatiri et al., ICDAR 2023)  
> and block-level annotation leaks entity boundaries (EC-FUNSD, arXiv:2402.02379).  
> Label it as a pipeline check on the slide.

In [ ]:
# ── Run Track A ─────────────────────────────────────────────────────────
# 3 seeds × 20 epochs on FUNSD standard split.
# Expected wall time on P100: ~40–60 min.

!python src/train.py --config configs/funsd_lilt.yaml

In [ ]:
# ── Inspect Track A results ─────────────────────────────────────────────
import json
from pathlib import Path

funsd_metrics_path = Path("results/funsd_lilt/metrics.json")
if funsd_metrics_path.exists():
    metrics = json.loads(funsd_metrics_path.read_text())
    print(f"Track A — LiLT on FUNSD")
    print(f"  mean F1: {metrics['mean_f1']:.4f} ± {metrics['std_f1']:.4f}")
    print(f"  target:  ~0.88")
    print()
    for run in metrics["runs"]:
        print(f"  seed {run['seed']}: F1={run['test']['f1']:.4f}  "
              f"best_val_F1={run['best_val_f1']:.4f}  "
              f"malformed_rate={run['malformed_tags']['rate']:.4f}")
        print(f"    cross_check: {run['cross_check']}")
else:
    print(f"No metrics found at {funsd_metrics_path} — did Track A run?")

In [ ]:
# ── Plot: FUNSD training curves ────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 150})

if funsd_metrics_path.exists():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    for run in metrics["runs"]:
        epochs = [h["epoch"] for h in run["history"]]
        losses = [h["train_loss"] for h in run["history"]]
        val_f1s = [h["val_f1"] for h in run["history"]]

        ax1.plot(epochs, losses, label=f"seed {run['seed']}")
        ax2.plot(epochs, val_f1s, label=f"seed {run['seed']}")

    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Train Loss")
    ax1.set_title("FUNSD · LiLT Training Loss")
    ax1.legend()

    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Validation F1")
    ax2.set_title("FUNSD · LiLT Validation F1")
    ax2.axhline(0.88, color="gray", linestyle="--", linewidth=0.8, label="target ≈ 0.88")
    ax2.legend()

    plt.tight_layout()
    fig.savefig("slides/figures/funsd_curve.pdf", bbox_inches="tight")
    print("Saved: slides/figures/funsd_curve.pdf")
    plt.show()
else:
    print("Skipping plot — no Track A metrics.")

---
## 3 · Track B — The Headline: VRDU MTL vs UTL Gap

**Purpose**: Measure the generalization gap with LiLT on VRDU Registration
Forms, leave-one-template-out, at matched 200-doc training size, ≥3 seeds.

**Calibration**: FormNet at 200 docs → MTL 90.51 vs UTL 77.29 (13.22-pt gap).

> This is the number that goes on the slide.  
> 3 templates → 3 UTL folds + 3 MTL folds, each with 3 seeds = 18 training runs.

In [ ]:
# ── Run Track B ─────────────────────────────────────────────────────────
# Matched MTL/UTL protocol: 3 folds × 2 regimes × 3 seeds × 20 epochs.
# Expected wall time on P100: ~3–5 hrs.

!python src/train.py --config configs/vrdu_lilt.yaml --protocol matched

In [ ]:
# ── Inspect Track B results ─────────────────────────────────────────────
vrdu_metrics_path = Path("results/vrdu_lilt/metrics.json")

if vrdu_metrics_path.exists():
    vrdu_metrics = json.loads(vrdu_metrics_path.read_text())
    gap = vrdu_metrics["gap"]

    print("Track B — LiLT on VRDU Registration Forms")
    print(f"  MTL mean F1: {gap['mtl_mean_f1']:.4f} ± {gap['mtl_std']:.4f}  ({gap['mtl_folds']} runs)")
    print(f"  UTL mean F1: {gap['utl_mean_f1']:.4f} ± {gap['utl_std']:.4f}  ({gap['utl_folds']} runs)")
    print(f"  GAP (MTL - UTL): {gap['gap_f1']:.4f}  ({gap['relative_drop_pct']:.1f}% relative drop)")
    print(f"  FormNet calibration gap: 13.22 points")
    print()

    # Per-fold breakdown
    print("Per-run detail:")
    for regime in ("MTL", "UTL"):
        runs = vrdu_metrics[regime]
        print(f"\n  {regime}:")
        for r in runs:
            held = r.get("held_out", [])
            held_str = ", ".join(held) if held else "(all)"
            print(f"    fold={r['fold']} seed={r['seed']} F1={r['test']['f1']:.4f}  "
                  f"held_out=[{held_str}]")
else:
    print(f"No metrics at {vrdu_metrics_path} — did Track B run?")

In [ ]:
# ── Plot: VRDU MTL vs UTL per-template bar chart ────────────────────────
import numpy as np

if vrdu_metrics_path.exists():
    # Aggregate F1 per fold (template) and regime
    from collections import defaultdict

    fold_f1 = defaultdict(lambda: defaultdict(list))  # regime -> fold -> [f1s]
    fold_labels = {}  # fold -> held_out template name

    for regime in ("MTL", "UTL"):
        for r in vrdu_metrics[regime]:
            fold_f1[regime][r["fold"]].append(r["test"]["f1"])
            if r["fold"] not in fold_labels and r.get("held_out"):
                fold_labels[r["fold"]] = r["held_out"][0]

    folds = sorted(fold_f1["MTL"].keys())
    template_names = [fold_labels.get(f, f"Fold {f}") for f in folds]

    mtl_means = [np.mean(fold_f1["MTL"][f]) for f in folds]
    mtl_stds  = [np.std(fold_f1["MTL"][f], ddof=1) if len(fold_f1["MTL"][f]) > 1 else 0 for f in folds]
    utl_means = [np.mean(fold_f1["UTL"][f]) for f in folds]
    utl_stds  = [np.std(fold_f1["UTL"][f], ddof=1) if len(fold_f1["UTL"][f]) > 1 else 0 for f in folds]

    x = np.arange(len(folds))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))
    bars_mtl = ax.bar(x - width/2, mtl_means, width, yerr=mtl_stds,
                      label="MTL (seen)", color="#4C72B0", capsize=4)
    bars_utl = ax.bar(x + width/2, utl_means, width, yerr=utl_stds,
                      label="UTL (unseen)", color="#DD8452", capsize=4)

    ax.set_xlabel("Held-out Template")
    ax.set_ylabel("Entity F1")
    ax.set_title("VRDU Registration Forms · LiLT · MTL vs UTL (n_train=200)")
    ax.set_xticks(x)
    ax.set_xticklabels(template_names, rotation=15, ha="right")
    ax.legend()
    ax.set_ylim(0, 1.05)

    # Annotate the overall gap
    overall_gap = gap["gap_f1"]
    ax.axhline(gap["mtl_mean_f1"], color="#4C72B0", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.axhline(gap["utl_mean_f1"], color="#DD8452", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.annotate(f"Gap: {overall_gap:.2f}",
                xy=(len(folds) - 0.3, (gap["mtl_mean_f1"] + gap["utl_mean_f1"]) / 2),
                fontsize=12, fontweight="bold", color="#333333",
                ha="left", va="center")

    plt.tight_layout()
    fig.savefig("slides/figures/vrdu_gap.pdf", bbox_inches="tight")
    print("Saved: slides/figures/vrdu_gap.pdf")
    plt.show()
else:
    print("Skipping plot — no Track B metrics.")

---
## 4 · Error Analysis

Classify prediction errors into the four-bucket failure taxonomy:
1. OCR / recognition errors
2. Layout-association errors (right text, wrong key)
3. Field-type / schema errors
4. Repeated-structure errors

This runs on the Track B results (UTL condition — the unseen-template errors
are the ones that matter for the project's story).

In [ ]:
# ── Error taxonomy on Track B UTL predictions ──────────────────────────
# Re-run inference on the best UTL checkpoint and classify errors.
# This requires the trained model state, so we re-load and predict.

from data.vrdu_loader import load_vrdu, load_splits, verify_template_recovery
from data.splits import matched_size_protocol, assert_no_template_leakage
from models.token_clf import TokenClassifier
from train import TrainConfig, set_seed, predict, FormDataset, make_collator, train_one
from evaluate import entity_f1
from errors import classify_errors, summarize_errors, error_table
from torch.utils.data import DataLoader

config = TrainConfig.from_yaml("configs/vrdu_lilt.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

examples, label_list, id2label = load_vrdu(Path(config.data_root), config.subcorpus)
protocol = matched_size_protocol(examples, n_train=config.n_train or 200, seed=config.seeds[0])

# Run error analysis on UTL folds only (the unseen-template condition)
all_preds, all_refs, all_words = [], [], []

for fold_idx, split in enumerate(protocol["UTL"]):
    print(f"\n--- UTL fold {fold_idx} (held-out: {split.held_out_templates}) ---")
    assert_no_template_leakage(split)

    seed = config.seeds[0]  # use first seed for error analysis
    set_seed(seed)

    classifier = TokenClassifier(config.backbone, len(label_list), id2label, config.max_length)
    result = train_one(classifier, split.train, split.val, split.test, config, device)
    print(f"  Test F1: {result['test']['f1']:.4f}")

    # Collect predictions for error analysis
    test_loader = DataLoader(
        FormDataset(split.test), batch_size=config.batch_size,
        collate_fn=make_collator(classifier),
    )
    preds, refs = predict(classifier, test_loader, device)
    words = [ex.words for ex in split.test]

    all_preds.extend(preds)
    all_refs.extend(refs)
    all_words.extend(words)

print(f"\nTotal test documents for error analysis: {len(all_preds)}")

In [ ]:
# ── Error taxonomy breakdown ────────────────────────────────────────────
breakdown = classify_errors(all_preds, all_refs, all_words, id2label)
print(summarize_errors(breakdown))

In [ ]:
# ── Error taxonomy table (for slides) ──────────────────────────────────
import pandas as pd

df = pd.DataFrame(error_table(breakdown))
print(df.to_string(index=False))
print()

# Pie chart for slides
fig, ax = plt.subplots(figsize=(6, 6))
fracs = breakdown.bucket_fractions()
labels = [k.replace("_", " ").title() for k in fracs]
sizes = list(fracs.values())
colors = ["#E24A33", "#348ABD", "#988ED5", "#FBC15E"]

# Only show non-zero buckets
filtered = [(l, s, c) for l, s, c in zip(labels, sizes, colors) if s > 0]
if filtered:
    labels_f, sizes_f, colors_f = zip(*filtered)
    ax.pie(sizes_f, labels=labels_f, colors=colors_f, autopct="%1.1f%%",
           startangle=90, textprops={"fontsize": 11})
    ax.set_title("Error Taxonomy — UTL Condition")
    plt.tight_layout()
    fig.savefig("slides/figures/error_taxonomy.pdf", bbox_inches="tight")
    print("Saved: slides/figures/error_taxonomy.pdf")
    plt.show()
else:
    print("No errors to plot (perfect predictions).")

---
## 5 · Collect Outputs

Download these from the Kaggle notebook output:
- `results/funsd_lilt/metrics.json` — Track A numbers
- `results/vrdu_lilt/metrics.json` — Track B numbers + gap
- `slides/figures/funsd_curve.pdf` — training curves for slide 19
- `slides/figures/vrdu_gap.pdf` — MTL vs UTL bar chart for slide 20
- `slides/figures/error_taxonomy.pdf` — error breakdown for slide 21

In [ ]:
# ── List all outputs ────────────────────────────────────────────────────
print("=== Generated files ===")
for pattern in ("results/**/*.json", "slides/figures/*.pdf"):
    for p in sorted(Path(".").glob(pattern)):
        print(f"  {p}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# ── Summary for the presentation ───────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY FOR SLIDES")
print("=" * 60)

if funsd_metrics_path.exists():
    m = json.loads(funsd_metrics_path.read_text())
    print(f"\nTrack A (pipeline check): LiLT on FUNSD")
    print(f"  F1 = {m['mean_f1']:.4f} ± {m['std_f1']:.4f}  (target ≈ 0.88)")

if vrdu_metrics_path.exists():
    m = json.loads(vrdu_metrics_path.read_text())
    g = m["gap"]
    print(f"\nTrack B (headline): LiLT on VRDU Registration Forms")
    print(f"  MTL (seen):   {g['mtl_mean_f1']:.4f} ± {g['mtl_std']:.4f}")
    print(f"  UTL (unseen): {g['utl_mean_f1']:.4f} ± {g['utl_std']:.4f}")
    print(f"  Gap:          {g['gap_f1']:.4f}  ({g['relative_drop_pct']:.1f}% drop)")
    print(f"  FormNet ref:  13.22 pt gap (MTL 90.51 / UTL 77.29)")

print("\n" + "=" * 60)